# RNG seeding control

This notebook uses `Procedural-FrozenLake-v1` because its generated maps are easy to inspect.

There are two mouse-env controls to understand:

- `kwargs={"map_seed": ...}` controls the generated FrozenLake board and goal rewards. This is specific to first-party procedural envs, so it is passed through `EnvConfig.kwargs`.
- `reset_seed` controls the seeds mouse-env passes to Gymnasium `env.reset(seed=...)` calls. In Gymnasium, reset seeding normally controls reset-time randomness such as initial state selection or randomized reset observations.

Random action sampling is ordinary Gymnasium behavior: `env.sample_random_inputs()` samples from the underlying action space. If you want to seed that, seed the per-env Gymnasium action space through `env.action_space.spaces[i]`.

The examples build up one idea at a time. In each section, only one seed is changed so you can see exactly what that seed controls.

In [ ]:
import json

from mouse_envs import EnvConfig, make_env


BASE_KWARGS = {
    "emit_map": True,
    "min_width": 4,
    "max_width": 4,
    "min_height": 4,
    "max_height": 4,
    "hole_prob": 0.25,
}


def board_from_output(output: dict) -> list[str]:
    return json.loads(output["info_map"])["board"]


def print_board(board: list[str]) -> None:
    print("\n".join(board))

## 1. `map_seed` controls the generated map

Intent: show that `map_seed` controls the generated FrozenLake board, while `reset_seed` does not.

`map_seed` is not a base mouse-env field. It belongs to `Procedural-FrozenLake-v1`, so it is passed through `kwargs`.

This example builds three envs:

- A: `map_seed=10`, `reset_seed=20`
- B: `map_seed=10`, `reset_seed=21`
- C: `map_seed=11`, `reset_seed=20`

Expected result: A and B should have the same board because `map_seed` is the same. C should have a different board because only `map_seed` changed.

In [ ]:
# Same map_seed, different reset seeds.
cfg_same_map_a = EnvConfig(
    id="Procedural-FrozenLake-v1",
    reset_seed=20,
    kwargs={**BASE_KWARGS, "map_seed": 10},
)
cfg_same_map_b = EnvConfig(
    id="Procedural-FrozenLake-v1",
    reset_seed=21,
    kwargs={**BASE_KWARGS, "map_seed": 10},
)

# Different map_seed, same reset seed as cfg_same_map_a.
cfg_different_map = EnvConfig(
    id="Procedural-FrozenLake-v1",
    reset_seed=20,
    kwargs={**BASE_KWARGS, "map_seed": 11},
)

env_same_map_a = make_env(cfg_same_map_a)
try:
    # The first step performs mouse-env's internal reset and returns info_map.
    output_same_map_a = env_same_map_a.step(env_same_map_a.sample_random_inputs())[0]
finally:
    env_same_map_a.close()

env_same_map_b = make_env(cfg_same_map_b)
try:
    output_same_map_b = env_same_map_b.step(env_same_map_b.sample_random_inputs())[0]
finally:
    env_same_map_b.close()

env_different_map = make_env(cfg_different_map)
try:
    output_different_map = env_different_map.step(env_different_map.sample_random_inputs())[0]
finally:
    env_different_map.close()

same_map_a = board_from_output(output_same_map_a)
same_map_b = board_from_output(output_same_map_b)
different_map = board_from_output(output_different_map)

print("A: map_seed=10, reset_seed=20")
print_board(same_map_a)
print()

print("B: map_seed=10, reset_seed=21")
print_board(same_map_b)
print()

print("C: map_seed=11, reset_seed=20")
print_board(different_map)
print()

print("A == B: same map_seed, so the boards match ->", same_map_a == same_map_b)
print("A != C: different map_seed, so the boards differ ->", same_map_a != different_map)

## 2. `reset_seed` controls reset-time randomness

Intent: show that `reset_seed` controls randomness that happens when an episode starts.

mouse-env passes `EnvConfig.reset_seed` into Gymnasium's `env.reset(seed=...)` internally because users keep calling `step()`. In normal Gymnasium envs, that seed controls the random number generator used for reset-time randomness: initial state sampling, randomized reset observations, and any other randomness used to start a new episode.

To make the effect visible, this section uses a fixed board with two `S` start tiles. There is no `map_seed` here because the map is not generated. The only thing changing is `reset_seed`.

Expected result: both envs use the same fixed board, but the first reset observation can differ because the reset seed changes the selected start tile.

In [ ]:
MULTI_START_BOARD = [
    "SFFS",
    "FFFF",
    "FFFF",
    "GFFG",
]

cfg_reset_seed_1 = EnvConfig(
    id="Procedural-FrozenLake-v1",
    reset_seed=1,
    kwargs={"emit_map": True, "fixed_map": MULTI_START_BOARD},
)
cfg_reset_seed_2 = EnvConfig(
    id="Procedural-FrozenLake-v1",
    reset_seed=2,
    kwargs={"emit_map": True, "fixed_map": MULTI_START_BOARD},
)

env_reset_seed_1 = make_env(cfg_reset_seed_1)
try:
    reset_a = env_reset_seed_1.step(env_reset_seed_1.sample_random_inputs())[0]
finally:
    env_reset_seed_1.close()

env_reset_seed_2 = make_env(cfg_reset_seed_2)
try:
    reset_b = env_reset_seed_2.step(env_reset_seed_2.sample_random_inputs())[0]
finally:
    env_reset_seed_2.close()

obs_a = reset_a.get("observation", reset_a.get("observation_discrete"))
obs_b = reset_b.get("observation", reset_b.get("observation_discrete"))

print("Fixed board with two possible starts (state 0 and state 3):")
print_board(MULTI_START_BOARD)
print()

print("A: reset_seed=1 -> first observation", int(obs_a.item()))
print("B: reset_seed=2 -> first observation", int(obs_b.item()))
print("Same fixed board in both outputs ->", board_from_output(reset_a) == board_from_output(reset_b))
print("Different reset_seed can select a different start ->", int(obs_a.item()) != int(obs_b.item()))